# Bab 7: Membangun Aplikasi Obrolan
## Panduan Cepat API OpenAI

Catatan ini diadaptasi dari [Repositori Contoh Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang berisi catatan yang mengakses layanan [Azure OpenAI](aoai-assignment.ipynb).

API OpenAI Python juga bekerja dengan Model Azure OpenAI, dengan beberapa modifikasi. Pelajari lebih lanjut tentang perbedaannya di sini: [Cara beralih antara endpoint OpenAI dan Azure OpenAI dengan Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Ikhtisar  
"Model bahasa besar adalah fungsi yang memetakan teks ke teks. Diberikan sebuah string input teks, model bahasa besar mencoba untuk memprediksi teks yang akan muncul selanjutnya"(1). Notebook "quickstart" ini akan memperkenalkan pengguna pada konsep LLM tingkat tinggi, persyaratan paket inti untuk memulai dengan AML, pengenalan ringan tentang desain prompt, dan beberapa contoh singkat dari berbagai kasus penggunaan. 


## Daftar Isi  

[Ikhtisar](#overview)  
[Cara menggunakan Layanan OpenAI](#how-to-use-openai-service)  
[1. Membuat Layanan OpenAI Anda](#1.-creating-your-openai-service)  
[2. Instalasi](#2.-installation)    
[3. Kredensial](#3.-credentials)  

[Kasus Penggunaan](#use-cases)    
[1. Meringkas Teks](#1.-summarize-text)  
[2. Mengklasifikasikan Teks](#2.-classify-text)  
[3. Membuat Nama Produk Baru](#3.-generate-new-product-names)  
[4. Menyesuaikan Klasifier](#4.fine-tune-a-classifier)  

[Referensi](#references)


### Bangun prompt pertama Anda  
Latihan singkat ini akan memberikan pengenalan dasar untuk mengirimkan prompt ke model OpenAI untuk tugas sederhana "ringkasan".


**Langkah-langkah**:  
1. Pasang perpustakaan OpenAI di lingkungan python Anda  
2. Muat perpustakaan pembantu standar dan atur kredensial keamanan OpenAI khas Anda untuk Layanan OpenAI yang telah Anda buat  
3. Pilih model untuk tugas Anda  
4. Buat prompt sederhana untuk model  
5. Kirim permintaan Anda ke API model!


### 1. Pasang OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Impor pustaka pembantu dan instansiasi kredensial


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Menemukan model yang tepat  
Model seperti GPT-4o dan GPT-4o mini dapat memahami dan menghasilkan bahasa alami, dan tersedia di platform OpenAI dengan berbagai tingkat kekuatan dan kecepatan yang sesuai untuk berbagai tugas.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Desain Prompt  

"Keajaiban model bahasa besar adalah dengan dilatih untuk meminimalkan kesalahan prediksi ini pada sejumlah besar teks, model akhirnya belajar konsep yang berguna untuk prediksi ini. Misalnya, mereka belajar konsep seperti"(1):

* cara mengeja
* cara kerja tata bahasa
* cara memparafrasekan
* cara menjawab pertanyaan
* cara mengadakan percakapan
* cara menulis dalam banyak bahasa
* cara membuat kode
* dll.

#### Cara mengendalikan model bahasa besar  
"Dari semua masukan ke model bahasa besar, yang paling berpengaruh adalah teks prompt(1).

Model bahasa besar dapat dipicu untuk menghasilkan keluaran dengan beberapa cara:

Instruksi: Beritahu model apa yang Anda inginkan
Penyelesaian: Membuat model menyelesaikan awal dari apa yang Anda inginkan
Demonstrasi: Tunjukkan kepada model apa yang Anda inginkan, baik dengan:
Beberapa contoh dalam prompt
Ratusan atau ribuan contoh dalam dataset pelatihan penyetelan halus"



#### Ada tiga pedoman dasar dalam membuat prompt:

**Tunjukkan dan beri tahu**. Jelaskan dengan jelas apa yang Anda inginkan baik melalui instruksi, contoh, atau kombinasi keduanya. Jika Anda ingin model mengurutkan daftar item secara alfabet atau mengklasifikasikan paragraf berdasarkan sentimen, tunjukkan bahwa itu yang Anda inginkan.

**Sediakan data berkualitas**. Jika Anda mencoba membangun pengklasifikasi atau membuat model mengikuti pola, pastikan ada cukup contoh. Pastikan untuk memeriksa kembali contoh Anda — model biasanya cukup pintar untuk melihat kesalahan ejaan dasar dan memberi Anda respons, tetapi juga mungkin menganggap ini disengaja dan dapat memengaruhi respons.

**Periksa pengaturan Anda.** Pengaturan temperature dan top_p mengontrol seberapa deterministik model dalam menghasilkan respons. Jika Anda meminta respons di mana hanya ada satu jawaban yang benar, maka Anda ingin mengatur ini lebih rendah. Jika Anda mencari respons yang lebih beragam, maka Anda mungkin ingin mengaturnya lebih tinggi. Kesalahan nomor satu yang dibuat orang dengan pengaturan ini adalah menganggap bahwa ini adalah kontrol "kecerdasan" atau "kreativitas".


Sumber: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Kirim!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Meringkas Teks  
#### Tantangan  
Meringkas teks dengan menambahkan 'tl;dr:' di akhir sebuah bagian teks. Perhatikan bagaimana model memahami cara melakukan sejumlah tugas tanpa instruksi tambahan. Anda dapat bereksperimen dengan prompt yang lebih deskriptif daripada tl;dr untuk memodifikasi perilaku model dan menyesuaikan ringkasan yang Anda terima(3).  

Pekerjaan terbaru telah menunjukkan peningkatan substansial pada banyak tugas dan tolok ukur NLP dengan melatih awal pada korpus teks yang besar diikuti dengan penyetelan khusus pada tugas tertentu. Meskipun biasanya arsitekturnya tidak bergantung pada tugas, metode ini masih memerlukan dataset penyetelan khusus tugas sebanyak ribuan atau puluhan ribu contoh. Sebaliknya, manusia umumnya dapat melakukan tugas bahasa baru hanya dengan beberapa contoh atau instruksi sederhana - sesuatu yang masih cukup sulit dilakukan oleh sistem NLP saat ini. Di sini kami menunjukkan bahwa meningkatkan skala model bahasa sangat meningkatkan performa tanpa tergantung tugas dan sedikit contoh, kadang-kadang bahkan mencapai kompetitif dengan pendekatan penyetelan sebelumnya yang terbaik di kelasnya.  



Ringkasannya  


# Latihan untuk beberapa kasus penggunaan  
1. Meringkas Teks  
2. Mengklasifikasikan Teks  
3. Menghasilkan Nama Produk Baru


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasifikasikan Teks  
#### Tantangan  
Klasifikasikan item ke dalam kategori yang diberikan saat waktu inferensi. Dalam contoh berikut, kami menyediakan kategori dan teks untuk diklasifikasikan dalam prompt(*playground_reference). 

Pertanyaan Pelanggan: Halo, salah satu tombol pada keyboard laptop saya baru-baru ini rusak dan saya memerlukan pengganti:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Buat Nama Produk Baru
#### Tantangan
Buat nama produk dari kata contoh. Di sini kami menyertakan dalam prompt informasi tentang produk yang akan kami buat namanya. Kami juga menyediakan contoh serupa untuk menunjukkan pola yang ingin kami terima. Kami juga telah mengatur nilai suhu tinggi untuk meningkatkan keacakan dan respons yang lebih inovatif.

Deskripsi produk: Pembuat milkshake rumahan
Kata benih: cepat, sehat, kompak.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Deskripsi produk: Sepasang sepatu yang dapat muat di segala ukuran kaki.
Kata benih: dapat disesuaikan, pas, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referensi  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Praktik terbaik untuk penyetelan GPT model agar dapat mengklasifikasikan teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lebih Lanjut  
[Tim Komersialisasi OpenAI](AzureOpenAITeam@microsoft.com) 


# Kontributor
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
